In [3]:
# Install only basic library
!pip install torch-geometric

import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import SAGEConv
import random

# Load dataset
dataset = Planetoid(root='/content/data', name='CiteSeer')
data = dataset[0]

# Model
conv1 = SAGEConv(dataset.num_features, 64)
conv2 = SAGEConv(64, dataset.num_classes)

optimizer = torch.optim.Adam(
    list(conv1.parameters()) + list(conv2.parameters()),
    lr=0.01
)

# --------- MANUAL SAMPLING FUNCTION ---------
def sample_nodes(mask, num_samples):
    indices = mask.nonzero(as_tuple=True)[0]
    perm = torch.randperm(len(indices))
    return indices[perm[:num_samples]]

# Training
for epoch in range(1, 11):
    optimizer.zero_grad()

    # SAMPLE 64 nodes (this is sampling)
    batch_nodes = sample_nodes(data.train_mask, 64)

    x = data.x
    edge_index = data.edge_index

    # Forward pass
    x = conv1(x, edge_index)
    x = F.relu(x)
    x = conv2(x, edge_index)

    # Loss only on sampled nodes
    loss = F.cross_entropy(
        x[batch_nodes],
        data.y[batch_nodes]
    )

    loss.backward()
    optimizer.step()

    print("Epoch:", epoch, "Loss:", loss.item())

# Testing
x = data.x
edge_index = data.edge_index

x = conv1(x, edge_index)
x = F.relu(x)
x = conv2(x, edge_index)

pred = x.argmax(dim=1)

correct = int((pred[data.test_mask] == data.y[data.test_mask]).sum())
accuracy = correct / int(data.test_mask.sum())
print("*"*100)
print("Final Accuracy:", accuracy)

Processing...
Done!


Epoch: 1 Loss: 1.7899366617202759
Epoch: 2 Loss: 1.3867343664169312
Epoch: 3 Loss: 0.7708076238632202
Epoch: 4 Loss: 0.5205010771751404
Epoch: 5 Loss: 0.29234153032302856
Epoch: 6 Loss: 0.08608739078044891
Epoch: 7 Loss: 0.024074029177427292
Epoch: 8 Loss: 0.009281898848712444
Epoch: 9 Loss: 0.004909256938844919
Epoch: 10 Loss: 0.00376352877356112
Final Accuracy: 0.5
